<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_68_workflow_orchestration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🔁 **Module 68 — Workflow Orchestration (Airflow / Prefect / Dagster)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 9 · Production Gaps


# Module 68 — Workflow Orchestration

> Every ML pipeline eventually has the same shape: **fetch raw data → clean → feature-engineer → train → evaluate → register → deploy**. Doing that once is a script. Doing it **every night at 3 AM with retries, alerts, lineage, and backfills** is a workflow orchestrator. This module is the map of the three Python options people actually use — **Airflow**, **Prefect**, and **Dagster** — plus the ML-native cousins (Argo Workflows, Kubeflow Pipelines, ZenML, Metaflow).

### What you'll cover
1. Why you need an orchestrator (the pain it removes)
2. The three core concepts: **DAGs, tasks, schedules**
3. **Airflow** — the original, operator-based, ubiquitous
4. **Prefect** — Python-first, dynamic, hosted-friendly
5. **Dagster** — asset-based, typed, observability-first
6. ML-native: **Argo Workflows, Kubeflow Pipelines, ZenML, Metaflow**
7. Retries, alerts, idempotency, backfills
8. Lineage + observability + cost
9. The simplest alternative — **cron + a single Python script**
10. Decision table — pick your orchestrator


## 1 · Why orchestrate

Without an orchestrator, a "production pipeline" looks like:

```
   crontab:  0 3 * * *  python /etc/scripts/train.py
```

Then:
- ❌ Task #3 of 7 failed at 3:04 AM → entire pipeline is broken, nobody knows until 9 AM standup.
- ❌ The dataset isn't there yet — task #2 ran on stale data. Silently.
- ❌ It worked yesterday but the input schema changed. No alert.
- ❌ You need to **rerun** the last 14 days because of a bug fix. Hand-rolling 14 cron jobs.
- ❌ Where did *this specific model* come from? Which run? Which dataset? No idea.

An orchestrator gives you:
- **Dependencies** — task B runs *only after* task A succeeded.
- **Retries** — flaky API → automatic backoff retry, page only if all retries fail.
- **Scheduling** — cron, plus interval, plus event-triggered.
- **Backfills** — "rerun every day from Mar 1 to Mar 14" is one command.
- **Lineage** — which dataset version produced this model.
- **Observability** — UI to see what ran, when, how long, why it failed.

You can build all of this yourself. You shouldn't.

## 2 · The three concepts every orchestrator shares

| Term | What it is |
|---|---|
| **Task** (Airflow) / **Task** (Prefect) / **Op** + **Asset** (Dagster) | a unit of work — a Python function that runs on a schedule |
| **DAG** (Directed Acyclic Graph) | the dependency graph between tasks — A → B → C |
| **Schedule** | when to start a DAG run — cron, interval, "at midnight UTC daily", or event-driven |
| **Sensor** / **Trigger** | wait for an external condition — a file landed in S3, a Kafka message, an HTTP webhook |
| **Run** | one execution of a DAG at a specific *logical date* |
| **Backfill** | re-run a DAG for past logical dates |
| **XCom** / **Result** / **Asset** | how data flows between tasks |

All three tools agree on these names. They diverge in *how* you express the DAG.

## 3 · Airflow — the original

Born at Airbnb in 2014; the default orchestrator across every other company. Run by Apache; managed offerings from **Astronomer**, **Cloud Composer (GCP)**, **MWAA (AWS)**, **Azure Data Factory** (similar concept).

In [ ]:
airflow_dag = '''
# dags/ml_pipeline.py
from datetime import datetime
from airflow import DAG
from airflow.operators.python import PythonOperator

def extract():
    print("fetched 1000 rows")

def transform():
    print("cleaned them")

def train(model_name="rf"):
    print(f"trained {model_name}")

with DAG(
    dag_id="ml_pipeline",
    start_date=datetime(2024, 1, 1),
    schedule="0 3 * * *",                    # 3 AM daily
    catchup=False,
    default_args={"retries": 3, "retry_delay": 300},   # 3 retries, 5 min apart
    tags=["ml"],
) as dag:
    t_extract   = PythonOperator(task_id="extract",   python_callable=extract)
    t_transform = PythonOperator(task_id="transform", python_callable=transform)
    t_train     = PythonOperator(task_id="train",     python_callable=train,
                                  op_kwargs={"model_name": "xgb"})

    t_extract >> t_transform >> t_train     # the >> operator defines the DAG
'''
print(airflow_dag)

**What Airflow gets right.**
- **Ubiquitous** — every cloud has a managed Airflow.
- **Huge operator library** — `PostgresOperator`, `S3ToRedshiftOperator`, `BigQueryOperator`, `KubernetesPodOperator` — wrappers for nearly every system.
- **Mature UI** — gantt, graph, logs, retry buttons all in one place.

**What Airflow gets wrong.**
- DAGs are defined **at parse time** — dynamic structure (e.g. "one task per file") is awkward (TaskGroup helps, not enough).
- Passing data between tasks goes through **XCom**, which is a small-payload key-value store. Big intermediate dataframes don't fit; you serialise to S3 manually.
- The Python is the **boundary** — task code runs in the worker process; you can't easily pip-install a dependency just for one task.

## 4 · Prefect — Python-first

In [ ]:
prefect_flow = '''
# flows/ml_pipeline.py
from prefect import flow, task
from datetime import timedelta

@task(retries=3, retry_delay_seconds=300)
def extract():
    print("fetched 1000 rows")
    return 1000

@task
def transform(n_rows: int):
    print(f"cleaned {n_rows} rows")
    return n_rows

@task
def train(n_rows: int, model_name: str = "xgb"):
    print(f"trained {model_name} on {n_rows} rows")

@flow(name="ml_pipeline")
def ml_pipeline():
    n = extract()
    n = transform(n)
    train(n, model_name="xgb")

# Deployment — registers schedule + infrastructure
if __name__ == "__main__":
    ml_pipeline.serve(
        name="prod",
        cron="0 3 * * *",
    )
'''
print(prefect_flow)

**What Prefect does differently.**
- **No DAG declaration** — the flow function *is* the DAG. Dependencies are inferred from how task return values flow.
- **Dynamic** — `for f in files: process.submit(f)` just works. The DAG shape can depend on runtime data.
- **Python types pass through** — no XCom, no S3 manual round-trip; results live in a configurable artefact store.
- **Cleanest local dev experience** of the three — `prefect server start` and you have a UI in 30 seconds.
- **`prefect-cloud`** is the canonical hosted offering (free tier).

## 5 · Dagster — asset-based

In [ ]:
dagster_assets = '''
# project/assets.py
from dagster import asset, AssetExecutionContext, AutoMaterializePolicy

@asset
def raw_rows():
    return list(range(1000))

@asset(deps=[raw_rows])
def clean_rows(raw_rows):
    return [r for r in raw_rows if r % 7 != 0]

@asset(auto_materialize_policy=AutoMaterializePolicy.eager())
def trained_model(clean_rows):
    print(f"training on {len(clean_rows)} rows")
    return "model_v3"

# project/definitions.py
from dagster import Definitions, ScheduleDefinition, define_asset_job
from . import assets

ml_job = define_asset_job("ml_pipeline", selection="*trained_model")
ml_sched = ScheduleDefinition(job=ml_job, cron_schedule="0 3 * * *")

defs = Definitions(assets=[assets.raw_rows, assets.clean_rows, assets.trained_model],
                   jobs=[ml_job], schedules=[ml_sched])
'''
print(dagster_assets)

**Dagster's pitch.** Forget DAGs. Define your **assets** (datasets, models, dashboards). The framework computes the DAG by reading dependencies between assets.

**Why this matters for ML.**
- An asset is a **typed, versioned, observable object**. The UI shows you when each was last materialised and what depends on it.
- "Re-materialise the model whenever the cleaned data changes" is **declarative** (`AutoMaterializePolicy.eager()`) — no manual scheduling.
- **Lineage** — for every model artefact, Dagster shows the upstream code commits + data versions that produced it.
- Best in class for **data-quality-driven** ML pipelines.

Cost: steeper learning curve. The team needs to internalise "assets, not jobs."


## 6 · ML-native orchestrators (when k8s is your home)

If your platform is Kubernetes (M46), three more options worth knowing:

| Tool | Mental model | When to pick |
|---|---|---|
| **Argo Workflows** | step DAG in YAML, each step is a container | k8s-native, no Python required for orchestration |
| **Kubeflow Pipelines** | Argo + Python SDK + experiment tracking | ML-first; tight Vertex AI integration |
| **Flyte** | typed, versioned, k8s-native; Lyft's open source | Python typed pipelines at scale; strong caching |
| **ZenML** | Python framework that *generates* DAGs for Airflow / Kubeflow / Argo / Sagemaker | "MLOps framework" — switch backends without rewriting pipelines |
| **Metaflow** | Netflix's library, single-file Python pipelines | rapid ML prototyping; first-class AWS Batch + Step Functions |

**Argo Workflows** in particular is the engine **underneath** Kubeflow Pipelines and many internal platforms. If you already use Argo CD for GitOps (M47), Argo Workflows is sitting right there.

In [ ]:
argo_yaml = '''
# argo-workflow.yaml — one workflow, two steps
apiVersion: argoproj.io/v1alpha1
kind: Workflow
metadata: { generateName: ml-pipeline- }
spec:
  entrypoint: pipeline
  templates:
  - name: pipeline
    dag:
      tasks:
      - name: extract
        template: run-script
        arguments: { parameters: [{name: cmd, value: "python extract.py"}] }
      - name: train
        dependencies: [extract]
        template: run-script
        arguments: { parameters: [{name: cmd, value: "python train.py"}] }
  - name: run-script
    inputs: { parameters: [{name: cmd}] }
    container:
      image: my-org/ml:1.3.0
      command: ["sh","-c","{{inputs.parameters.cmd}}"]
'''
print(argo_yaml)

## 7 · Retries, alerts, idempotency, backfills

Five concerns every orchestrator must solve:

### Retries
- Set a default `retries=3, retry_delay=5min` per task.
- **Exponential backoff** for flaky external APIs.
- **Don't** retry on user-error (wrong schema) — fail fast.

### Alerts
- Slack / PagerDuty integration on DAG failure.
- Page only on **second** failure of the same DAG to avoid being woken by transient flakes.
- Include the **logical date** in the alert — "what was it trying to do?"

### Idempotency
- Every task should be **safe to re-run**. Output to a path that includes the logical date: `s3://bucket/dt=2025-01-15/`.
- **Never** append to the same file from two different runs.
- For database upserts, use `INSERT ... ON CONFLICT UPDATE` not blind `INSERT`.

### Backfills
- "Run 30 days of history" is a single command in all three tools.
- Limit **concurrency** of backfills — running 30 days in parallel will DDoS your data source.

### Schedule semantics
- Airflow uses **logical date** (the start of the period the run *represents*).
- "Daily at 3 AM" means the 2025-01-15 run starts at 2025-01-16 03:00 — it processes the day *that just ended*.
- Misunderstanding this is the #1 Airflow bug.

## 8 · Lineage + observability + cost

**Lineage** — what data + code produced this artefact.
- Dagster has it built in (asset graph).
- Airflow: bolt on **OpenLineage** → emit events to **Marquez** / **DataHub** / **Atlan**.
- Prefect: similar via OpenLineage integration.

**Observability** — what ran, how long, why it failed.
- The UI of each tool covers the basics.
- For deep production: send metrics to **Prometheus** (M50), traces to **OTel** (M51).

**Cost** — who paid for that run.
- Tag every task / pod with `team`, `env`, `pipeline` (M46 cost-allocation tags).
- Long-running training jobs are the #1 ML cost; orchestrators that show wall-clock per task are non-negotiable.

The 2025 lineage standard is **OpenLineage**. Every modern orchestrator can emit it.

## 9 · The simplest alternative — cron + a single script

Before reaching for Airflow, ask: **is your pipeline three tasks that run nightly and never branch?**

```
   crontab:  0 3 * * *  /opt/ml/run_nightly.sh
```

with `run_nightly.sh`:
```bash
set -euo pipefail
python extract.py >> /var/log/extract.log 2>&1 || { slack 'extract failed'; exit 1; }
python train.py   >> /var/log/train.log   2>&1 || { slack 'train failed';   exit 1; }
python deploy.py  >> /var/log/deploy.log  2>&1 || { slack 'deploy failed';  exit 1; }
```

You **don't** need an orchestrator if:
- Pipeline is linear (no branching).
- No retries / backfills needed.
- One team owns the whole thing.
- Failure can wait until morning.

You **do** need one if:
- More than ~5 tasks with non-trivial dependencies.
- Multiple teams depending on shared data.
- Backfills happen.
- Failure must page.

Reach for the orchestrator at the point the bash file would be longer than 50 lines, not before.

## 10 · Decision table — pick your orchestrator

| You are… | Pick |
|---|---|
| Already an Airflow shop with managed Composer / MWAA | **Airflow** — don't fix what isn't broken |
| Greenfield Python team, want clean local dev + dynamic flows | **Prefect** |
| Data-platform team wanting lineage + asset versioning | **Dagster** |
| Kubernetes-first ML platform, comfortable with YAML | **Argo Workflows** (often + Kubeflow) |
| ML team wanting one framework that targets Airflow/Kubeflow/SageMaker | **ZenML** |
| Two engineers, simple nightly pipelines | **cron + bash + Slack webhook** |
| Inside AWS, want AWS-managed everything | **Step Functions** + **EventBridge** (not Python-native, but native to AWS) |
| Heavy on **dbt** for the transform layer | **dbt Cloud** for transforms + **Dagster** to orchestrate |
| Streaming, not batch | **Flink** / **Kafka Streams** — orchestrators are for batch |

### Anti-patterns
- ❌ **Airflow as a general task queue.** Use **Celery** / **RQ** / **SQS** for ad-hoc work. Airflow's overhead per task is heavy.
- ❌ **Putting model training code in the DAG.** The DAG should *call* a container that runs the training — keep orchestration thin.
- ❌ **State in XCom.** XCom is for small artefacts (paths, ids). Big dataframes go to S3/GCS/Postgres.
- ❌ **No idempotency.** Two concurrent runs of the same logical date corrupting the output table is a classic.
- ❌ **Manual UI clicks instead of git.** Schedule definitions live in code; reviews happen in PRs.

## ✅ Recap
- Orchestrators give you **dependencies, retries, schedules, backfills, lineage, observability**.
- **Airflow** is the ubiquitous default; **Prefect** is the Python-first newcomer; **Dagster** centres on **typed assets**.
- ML-native cousins: **Argo Workflows, Kubeflow Pipelines, ZenML, Metaflow, Flyte**.
- **Cron + bash** is fine for <5 linear tasks; reach for an orchestrator when the bash file passes ~50 lines.
- Idempotency, lineage (OpenLineage), and cost tags are non-negotiable in production.
- Pick your tool from your team's strengths, not from Twitter trends.

Next: **M69 — Feature Stores (Feast)**.
